# Digit Recognizer (Library Version)

Notebook nay la ban sao tu `better.ipynb`, nhung su dung thu vien `scikit-learn` thay vi tu code forward/backprop.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Duong dan Kaggle giong notebook goc, kem fallback local
KAGGLE_TRAIN_PATH = '/kaggle/input/competitions/digit-recognizer/train.csv'
LOCAL_TRAIN_PATH = 'train.csv'

if os.path.exists(KAGGLE_TRAIN_PATH):
    data = pd.read_csv(KAGGLE_TRAIN_PATH)
elif os.path.exists(LOCAL_TRAIN_PATH):
    data = pd.read_csv(LOCAL_TRAIN_PATH)
else:
    raise FileNotFoundError('Khong tim thay train.csv trong Kaggle path hoac local path.')

data.head()

In [ ]:
X = data.drop(columns=['label']).values.astype(np.float32) / 255.0
y = data['label'].values

# Tach tap train/validation de danh gia mo hinh
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, random_state=42, stratify=y
)

print('X_train:', X_train.shape)
print('X_val  :', X_val.shape)

In [ ]:
# MLPClassifier tuong duong y tuong mang NN 1 hidden layer
clf = MLPClassifier(
    hidden_layer_sizes=(128,),
    activation='relu',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=20,
    random_state=42,
    verbose=True
)

clf.fit(X_train, y_train)

In [ ]:
val_pred = clf.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)

print(f'Validation accuracy: {val_acc:.4f}')
print('\nClassification report:')
print(classification_report(y_val, val_pred))

In [ ]:
def test_prediction(index, X_data, y_data, model):
    image_flat = X_data[index]
    prediction = model.predict(image_flat.reshape(1, -1))[0]
    label = y_data[index]

    print('Prediction:', prediction)
    print('Label     :', label)

    image_2d = image_flat.reshape(28, 28)
    plt.figure(figsize=(4, 4))
    plt.imshow(image_2d, cmap='gray')
    plt.axis('off')
    plt.show()


test_prediction(0, X_val, y_val, clf)
test_prediction(1, X_val, y_val, clf)
test_prediction(2, X_val, y_val, clf)

In [ ]:
# (Tuy chon) Train tren full train va du doan test.csv de nop Kaggle
KAGGLE_TEST_PATH = '/kaggle/input/competitions/digit-recognizer/test.csv'
LOCAL_TEST_PATH = 'test.csv'

test_path = KAGGLE_TEST_PATH if os.path.exists(KAGGLE_TEST_PATH) else LOCAL_TEST_PATH

if os.path.exists(test_path):
    X_test = pd.read_csv(test_path).values.astype(np.float32) / 255.0

    final_clf = MLPClassifier(
        hidden_layer_sizes=(128,),
        activation='relu',
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=20,
        random_state=42,
        verbose=True
    )
    final_clf.fit(X, y)

    test_pred = final_clf.predict(X_test)
    submission = pd.DataFrame({
        'ImageId': np.arange(1, len(test_pred) + 1),
        'Label': test_pred
    })
    submission.to_csv('submission_sklearn.csv', index=False)
    print('Da tao file submission_sklearn.csv')
else:
    print('Khong tim thay test.csv, bo qua buoc tao submission.')